<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/clip_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
import gc

PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
SAVE_DIR        = '/content/drive/MyDrive/only_text_clip_NEW'
MODEL_NAME      = 'clip_text_only_model_NEW'

# --- WEITERTRAINIEREN (RESUME) ---
CHECKPOINT_PATH = None

# --- HYPERPARAMETER ---
BATCH_SIZE    = 256
EPOCHS        = 20
LEARNING_RATE = 1e-5    # Standard-Lernrate für CLIP
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR TEXT MIT CLIP)
# ==========================================
class ClipTextDataset(Dataset):
    def __init__(self, json_path, processor):
        self.processor = processor
        self.samples = []
        self.label_map = {}
        self.id_to_label = {}

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Labels ermitteln (alphabetisch sortieren!)
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        for idx, label in enumerate(unique_labels):
            self.label_map[label] = idx
            self.id_to_label[idx] = label

        # Map speichern (Wichtig für die spätere Evaluation)
        with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
            json.dump(self.id_to_label, f)

        for item in tqdm(raw_data, desc="Lade Texte"):
            label_str = item.get('label')

            # --- TEXT HOLEN ---
            ocr_text = item.get('text', "").strip()
            # RAM-Trick: CLIP verarbeitet maximal 77 Token (ca. 300 Zeichen)
            if len(ocr_text) > 300: ocr_text = ocr_text[:300]
            if len(ocr_text) < 2: ocr_text = "meme text"

            self.samples.append({
                'text': ocr_text,
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} Texte geladen.\n")
        del raw_data
        gc.collect()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Processor NUR für Text aufrufen
        text_inputs = self.processor(
            text=[sample['text']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ==========================================
# 3. ARCHITEKTUR (CLIP TEXT-ONLY)
# ==========================================
class ClipTextOnlyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Backbone einfrieren
        for param in self.clip.parameters():
            param.requires_grad = False

        # Nur den letzten Layer des TEXT-Encoders auftauen (Partial Unfreezing)
        for param in self.clip.text_model.encoder.layers[-1].parameters():
            param.requires_grad = True
        for name, param in self.clip.named_parameters():
            if "layer_norm" in name:
                param.requires_grad = True

        # Classifier Head: Input ist 512 (Standard-Embedding-Größe von CLIP)
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        # Text durch den CLIP Text-Encoder jagen
        text_out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)

        # Projektion anwenden, um den finalen 512-Vektor zu bekommen
        text_embeds = self.clip.text_projection(text_out[1])

        # Klassifizieren
        return self.classifier(text_embeds)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
def run_clip_text_training():
    print("--- Start CLIP Text-Only Training ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Datasets aus den sauberen JSONs laden
    train_dataset = ClipTextDataset(PATH_TRAIN_JSON, processor)
    val_dataset   = ClipTextDataset(PATH_VAL_JSON, processor)

    num_classes = len(train_dataset.label_map)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ClipTextOnlyNet(num_classes).to(DEVICE)

    # Standard Optimizer mit einheitlicher Lernrate
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_acc = 0.0

    # --- LADE LOGIK FÜR FULL CHECKPOINT ---
    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint["epoch"]
            best_acc = checkpoint.get("best_acc", 0.0)
            print(f"[SUCCESS] Modell, Optimizer und Scheduler geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)
            print("[WARNUNG] Alter Checkpoint (Nur Gewichte). Optimizer fängt bei 0 an.")

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        # --- TRAINING ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # VRAM aufräumen
        del input_ids, attention_mask, labels, logits
        gc.collect()
        torch.cuda.empty_cache()

        # --- VALIDIERUNG ---
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(input_ids, attention_mask)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del input_ids, attention_mask, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat E{epoch+1}: Val Acc: {val_acc:.2%}")

        # --- CHECKPOINT SPEICHERN ---
        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc
        }

        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pth"))
            print(f"    [SAVED] Neues bestes Modell ({val_acc:.2%})!")

if __name__ == "__main__":
    run_clip_text_training()

--- Start CLIP Text-Only Training ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Epoche 19 [Valid]:   0%|          | 0/37 [00:00<?, ?it/s]

 -> Resultat E19: Val Acc: 42.47%
    [SAVED] Neues bestes Modell (42.47%)!


Epoche 20/20 [Train]:   0%|          | 0/296 [00:00<?, ?it/s]

Epoche 20 [Valid]:   0%|          | 0/37 [00:00<?, ?it/s]

 -> Resultat E20: Val Acc: 42.65%
    [SAVED] Neues bestes Modell (42.65%)!


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
from google.colab import drive

# ==========================================
# 1. KONFIGURATION
# ==========================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- PFADE (NUR DAS TEST-SET!) ---
PATH_TEST_JSON   = '/content/drive/MyDrive/meme_test.json' # Bitte verifizieren ob dies das richtige Test Set ist

# NEUE PFADE WIE IM TRAININGSSKRIPT
SAVE_DIR        = '/content/drive/MyDrive/only_text_clip_NEW'
CHECKPOINT_PATH = '/content/drive/MyDrive/only_text_clip_NEW/clip_text_only_model_NEW_best.pth'
PATH_LABEL_MAP  = '/content/drive/MyDrive/only_text_clip_NEW/clip_text_only_model_NEW_map.json'
OUTPUT_CSV      = 'clip_text_only_NEW_evaluation_results.csv'

BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR TEST-DATEN)
# ==========================================
class EvalClipTextDataset(Dataset):
    def __init__(self, json_path, label_map_path, processor):
        self.processor = processor
        self.samples = []

        # --- LADE DIE OFFIZIELLE LABEL MAP ---
        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            # json wandelt int-Keys in Strings um, wir wandeln sie zurück
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        print(f"--- [EVAL DATASET] Lade Test-Texte aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Texte"):
            label_str = item.get('label')

            # Unbekannte Klassen im Testset ignorieren (Sicherheitscheck)
            if label_str not in self.label_map:
                continue

            filename = item.get('filename', 'unknown_file')

            # --- TEXT HOLEN UND BEREINIGEN ---
            ocr_text = item.get('text', "").strip()
            if len(ocr_text) > 300: ocr_text = ocr_text[:300]
            if len(ocr_text) < 2: ocr_text = "meme text"

            self.samples.append({
                'text': ocr_text,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} Test-Texte geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        text_inputs = self.processor(
            text=[sample['text']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename'],
            'raw_text': sample['text']
        }

# ==========================================
# 3. ARCHITEKTUR (MUSS IDENTISCH SEIN)
# ==========================================
class ClipTextOnlyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        text_out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = self.clip.text_projection(text_out[1])
        return self.classifier(text_embeds)

# ==========================================
# 4. EVALUATION LOOP
# ==========================================
def run_clip_text_evaluation():
    print("--- Starte CLIP Text-Only Test-Evaluation ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    eval_dataset = EvalClipTextDataset(PATH_TEST_JSON, PATH_LABEL_MAP, processor)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    num_classes = len(eval_dataset.label_map)
    model = ClipTextOnlyNet(num_classes).to(DEVICE)

    # --- GEWICHTE LADEN ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        # Falls es ein Full-Checkpoint ist
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Berechne Vorhersagen auf ungesehenen Daten...")

    with torch.no_grad():
        for batch in tqdm(eval_loader):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            filenames = batch['filename']
            raw_texts = batch['raw_text']

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                # Bereinige den Text für die CSV
                clean_text = raw_texts[i].replace("\n", " ").replace(";", ",")

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2),
                    "OCR_Text": clean_text
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"CLIP TEXT-ONLY ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    df_details = pd.DataFrame(results_list)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "clip_text_only_NEW_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert.")

if __name__ == "__main__":
    run_clip_text_evaluation()


--- Starte CLIP Text-Only Test-Evaluation ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Lade Checkpoint: /content/drive/MyDrive/only_text_clip_NEW/clip_text_only_model_NEW_best.pth
[SUCCESS] Gewichte erfolgreich geladen.
Berechne Vorhersagen auf ungesehenen Daten...


  0%|          | 0/38 [00:00<?, ?it/s]


CLIP TEXT-ONLY ACCURACY (Test Set): 41.79%

[FERTIG] Tabellen gespeichert.


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
